In [1]:
# Define data to be processed
results_root = 'C:\\Users\\espen\\Documents\\work\\PhD\\papers\\ICIP_salmon_reid\\data\\tracking\\associator\\BoostCompTrack\\output\\salmon_tracking\\'
salmon_margin = 40
out_root = 'C:\\Users\\espen\\Documents\\work\\PhD\\papers\\ICIP_salmon_reid\\data\\reid\\'

verbose = False
size_thresh = 600
num_thresh = 20

In [ ]:
import yaml
import sys
# See https://github.com/espenbh/BoostCompTrack.
tracking_base = 'C:\\Users\\espen\\Documents\\work\\PhD\\papers\\2_salmon_tracking\\code\\salmon_component_tracking\\'
sys.path.append(tracking_base)
sys.path.append(tracking_base + 'helpers')
sys.path.append(tracking_base + 'associator')
sys.path.append(tracking_base + 'associator\\CompTrack')
sys.path.append(tracking_base + 'associator\\BoostTrack')
sys.path.append(tracking_base + 'associator\\BoostTrack\\external')

from file_utils import load_txt_data_for_eval
from comp_utils import get_salmon_ID_and_comp_type_from_comp_ID
from hq_track_utils import remove_incomplete_salmon, remove_small_salmon, remove_non_consecutive_tracks
from data_generation_helpers import subsample_trackers, create_and_save_labels, save_salmon_images

import numpy as np
from pathlib import Path




ERROR Error reading from C:\Users\espen\AppData\Roaming\Ultralytics\settings.json: "No Ultralytics setting 'openvino_msg'. \nView Ultralytics Settings with 'yolo settings' or at 'C:\\Users\\espen\\AppData\\Roaming\\Ultralytics\\settings.json'\nUpdate Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings."
Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\espen\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
all_analysis_folders = list(Path(results_root).iterdir())

for results_path in all_analysis_folders:
    #if "analysis7" in str(results_path):
    #   print(results_path)
    #else:
    #   continue

    results_path = str(results_path)
    config = yaml.safe_load(open(results_path + '\\config.yml', 'r'))
    analysis_path = Path(out_root) / Path(results_path).name 
    Path(Path(out_root) / Path(results_path).name).mkdir(exist_ok=True)
    Path(Path(out_root) / Path(results_path).name / Path('images')).mkdir(exist_ok=True)
    Path(Path(out_root) / Path(results_path).name / Path('labels')).mkdir(exist_ok=True)

    def save_trackers_as_txt(trackers, out_path):
        np.savetxt(out_path, trackers, fmt='%s', delimiter=',')

    refined_path = Path(analysis_path) / f"{Path(results_path).name}_results_refined.txt"

    if refined_path.exists():
        trackers = np.loadtxt(analysis_path / (Path(results_path).name + '_results_refined.txt'), dtype=str, delimiter=',')
    else:
        trackers = load_txt_data_for_eval(results_path + '\\MOT_results.txt')[:,:11]
        trackers[:,1],_ = get_salmon_ID_and_comp_type_from_comp_ID(trackers[:,1].astype(float).astype(int),9)
        trackers, id_cnt_dict = remove_incomplete_salmon(trackers, config, verbose=verbose, ncomp = 9, size_thresh = size_thresh)
        trackers = remove_small_salmon(trackers, config, id_cnt_dict, num_thresh=num_thresh, verbose=verbose)
        trackers, valid_ids = remove_non_consecutive_tracks(trackers, num_thresh=num_thresh,verbose=verbose)
        trackers = subsample_trackers(trackers, valid_ids, subsample_rate=5, start_offset=2)
        save_trackers_as_txt(trackers, analysis_path / (Path(results_path).name + '_results_refined.txt'))
    save_salmon_images(trackers, config, salmon_margin, analysis_path)
    create_and_save_labels(trackers, results_path, out_root, "C:\\Users\\espen\\Documents\\work\\PhD\\papers\\ICIP_salmon_reid\\data\\segmentation\\yolo_annotated_salmon_patches\\yolo_segmentation_runs\\train\\weights\\best.pt", config)


0: 224x640 1 Q1, 1 Q2, 1 operculum, 90.2ms
Speed: 1.9ms preprocess, 90.2ms inference, 59.8ms postprocess per image at shape (1, 3, 224, 640)

0: 224x640 1 Q1, 1 Q2, 1 operculum, 11.9ms
Speed: 0.8ms preprocess, 11.9ms inference, 2.0ms postprocess per image at shape (1, 3, 224, 640)

0: 192x640 1 Q1, 2 Q2s, 1 operculum, 90.8ms
Speed: 0.8ms preprocess, 90.8ms inference, 1.9ms postprocess per image at shape (1, 3, 192, 640)

0: 192x640 1 Q1, 1 Q2, 1 operculum, 10.6ms
Speed: 1.0ms preprocess, 10.6ms inference, 1.8ms postprocess per image at shape (1, 3, 192, 640)

0: 192x640 1 Q1, 1 Q2, 1 operculum, 10.6ms
Speed: 0.9ms preprocess, 10.6ms inference, 1.6ms postprocess per image at shape (1, 3, 192, 640)

0: 192x640 2 Q1s, 2 Q2s, 2 operculums, 10.7ms
Speed: 1.3ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 192, 640)

0: 192x640 1 Q1, 1 Q2, 1 operculum, 10.7ms
Speed: 0.9ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 192, 640)

0: 192x